[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/chapters/9/notebook.ipynb)

# Chapter 9 · Attention & the Transformer
### Opening the BERT box, building a GPT

Two earlier chapters left you with a **promise** and a **mystery**.

- **Chapter 3 (the promise).** Your bigram model generated names from *one* previous character, then hit a wall — it could only remember one token. The lesson promised: *"that's where transformers come in, but that's a story for later chapters."* This is that chapter.
- **Chapter 8 (the mystery).** You watched BERT give the word *"bank"* a different vector in *"river bank"* vs *"money bank"*. A contextual embedding, it said, is **computed**, not looked up — but the computation was magic. This chapter opens that box.

Both have the same answer: **attention**. We build it from scratch, and every claim below is backed by a runnable check or a figure. By the end you'll have the exact machine inside both BERT (understands) and GPT (generates) — and see that a *single line* separates them.

| Model | Context it uses | How it combines context |
|---|---|---|
| Bigram (Ch 3) | the **1** previous token | a lookup in a V×V table |
| MLP (Ch 4) | a **fixed** window | learned weights on a fixed input |
| Transformer (**here**) | **all** tokens, any length | attention — a learned, data-dependent average |

---
## First, the big picture

Before we build anything, let's answer three questions: *what is attention*, *what is a transformer*, and *where did they come from*. Five minutes of orientation so the code that follows feels inevitable rather than magic.

### What is attention?

Read this sentence: *"I poured water from the pitcher until **it** was full."* You know **it** means the pitcher, not the water — and you knew it instantly, because you unconsciously **looked back** at the other words and judged which ones matter for understanding "it." Change one word — *"until it was **empty**"* — and suddenly "it" is the pitcher being emptied. Same word, different neighbors, different meaning.

**Attention is that "looking back," turned into arithmetic.** For every token, the model scores every *other* token — *"how relevant are you to me right now?"* — turns those scores into weights that sum to 1, and mixes the tokens together according to the weights. A token's new representation is a **blend of the tokens it chose to attend to**. That is the entire idea. Everything in this chapter is just *how* the scores get computed and *how* the blend is done efficiently.

### What is a transformer?

Attention lets tokens **share** information, but sharing isn't thinking. A **transformer** is the full architecture you get when you pair attention with the ordinary neural network from Chapter 4 and repeat:

> **attention** (tokens look at each other) → **a Chapter-4 network** (each token thinks about what it gathered) → do it again, several times.

Wrap those two steps with a little plumbing — residual connections and normalization to keep deep stacks trainable, and a *positional* signal so the model knows word order — and stack the pair N times. That stack is a transformer. **Attention gathers; the feed-forward net thinks; depth lets it build up meaning.** You already own every ingredient; this chapter just assembles them.

### A little history — why this was a big deal

- **Before 2017 (RNNs / LSTMs).** Models read text strictly left-to-right, one word at a time, cramming everything seen so far into a single fixed-size memory vector. Two problems: long-range memory *decayed* (by the end of a paragraph the opening was a blur), and the strict left-to-right dependency made them **slow** — you couldn't process word 50 until you'd finished word 49, so GPUs sat idle.
- **2014–2015 (attention as a helper).** Researchers bolted an early form of attention *onto* those recurrent models for machine translation, letting the model look back at all input words instead of one squished summary. Translation quality jumped.
- **2017 — "Attention Is All You Need" (Vaswani et al.).** The radical move: **throw the recurrence away and keep only attention.** No more reading one word at a time — every token attends to every other token *in parallel*. This both fixed the long-range memory problem and unlocked GPU-scale training. This paper is the transformer, and it's the architecture we build below.
- **2018 onward — BERT and GPT.** Two descendants split off almost immediately: **BERT** (attends in *both* directions → understands text, Chapter 8) and **GPT** (attends only *backward* → generates text). We'll see in **Part 5** that the only difference between them is a single line of code — a mask.
- **2018 onward, until very recently — the age of scaling.** For roughly half a decade the recipe barely changed: take the 2017 architecture and make it *bigger* — more layers, more data, more compute — giving GPT-2, GPT-3, GPT-4, and essentially every large language model you've used. The machine you build in this notebook is, structurally, the machine behind ChatGPT. Just smaller.
- **Today.** The frontier is moving *past* pure softmax attention, whose cost grows with the sequence length (O(T²) compute and a KV-cache that never stops growing). A wave of **linear-attention** and **state-space** methods swaps that ever-growing cache for a fixed-size recurrent **state** — giving **linear time and linear memory** — which is really the RNN idea coming back. This isn't fringe research: **Gated DeltaNet (GDN)** ships in Alibaba's **Qwen** and **Kimi Delta Attention (KDA)** in Moonshot's **Kimi**, both as hybrids that keep a few full-attention layers, alongside pure state-space models like **Mamba**. We build the classic softmax transformer below — the thing every one of these is measured against.

### You're already equipped

Nothing here is imported from nowhere — you built it all in earlier chapters:

| Ingredient | Where you met it |
|---|---|
| dot products & **softmax** | Chapter 2 (cross-entropy) |
| a **2-layer network** + gradient descent | Chapter 4 |
| **embeddings** (static, then contextual) | Chapters 6 & 8 |
| the **bigram baseline** & name generation | Chapter 3 |

Keep that table in mind. When a piece of the transformer shows up, we'll point back to exactly where you already learned it. Now let's build.

In [ ]:
# On Colab this installs the course package; locally it is a no-op.
import sys
if 'google.colab' in sys.modules:
    !pip install -q "git+https://github.com/BaytAlhikmah/hands-on-llms-for-swes.git#subdirectory=pkg"

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from alhikmah_llms import Chapter9

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

def check(msg, cond, value=None):
    "Print a claim with a ✓/✗ and (optionally) the value that proves it."
    mark = "✓" if cond else "✗ FAILED"
    print(f"CLAIM {mark}  {msg}" + (f"   →  {value}" if value is not None else ""))
    assert cond, msg

print("device =", DEVICE, "| torch", torch.__version__)

---
## Part 1 · A contextual embedding is a weighted average

The single most important sentence in this chapter:

> **A contextual embedding is a weighted average of the embeddings of the tokens around it.**

The static embedding of *"bank"* (a plain lookup, like Chapter 6's word2vec) is identical in both sentences. What makes it *contextual* is blending in a bit of *"river"* or a bit of *"money"*. "Blend in a bit of" **is** a weighted average. The whole chapter is one question: **what should the weights be?**

**🔮 Predict first.** We start with one static *"bank"* vector. If we mix it 50/50 with *"river"* in one sentence and 50/50 with *"money"* in another, will the two results be closer together or farther apart than the (identical) static vectors were?

In [ ]:
river = np.array([-2.0, 0.0])                      # (2,)  a 2-D word vector
money = np.array([2.0, 0.0])                       # (2,)
bank  = np.array([0.0, 2.0])                       # (2,)
mix = lambda a, b, w: (1 - w) * a + w * b          # weighted average of two (2,) vecs -> (2,)

bank_river = mix(bank, river, 0.5)                 # (2,)  'bank' in "... river bank"
bank_money = mix(bank, money, 0.5)                 # (2,)  'bank' in "... money bank"

check("static 'bank' is identical in both sentences",
      np.linalg.norm(bank - bank) == 0, "dist = 0.00")
check("mixing pushes the two 'bank' vectors apart",
      np.linalg.norm(bank_river - bank_money) > 1, f"dist = {np.linalg.norm(bank_river-bank_money):.2f}")

Chapter9.plot_vectors_2d(
    {"river": tuple(river), "money": tuple(money), "bank (static)": tuple(bank),
     "bank+river": tuple(bank_river), "bank+money": tuple(bank_money)},
    title="One static 'bank' → two contextual 'bank's after a weighted average")
plt.show()

---
## Part 2 · The dumbest possible attention (uniform averaging)

Before we make the weights *smart*, let's make them *stupid* — every token weighted equally — and prove even that does something. For each position we average that token with all tokens **before** it: position 0 sees only itself; position 4 sees the mean of tokens 0–4. Each token becomes a running summary of its past.

**🔮 Predict first.** We'll compute this two ways — a Python loop, and a single matrix multiply `W @ X`. What does row 0 of `W` look like? Row 4? Why is `W` lower-triangular rather than full?

**The trick (Karpathy's "mathematical trick in self-attention"):** *a weighted average over a sequence is a matrix multiply.* Uniform causal averaging is just the special case where `W` is lower-triangular with equal, row-summing-to-1 weights. Everything from here replaces this fixed `W` with one the model computes itself.

In [ ]:
T, d = 6, 4
X = torch.randn(T, d)                              # (T, d) = (6, 4)  — T token vectors of width d

# (a) the obvious way: a loop
loop = torch.stack([X[:t+1].mean(0) for t in range(T)])  # X[:t+1] (t+1,d) -> mean(0) (d,); stack -> (T, d)

# (b) the fast way: one matrix multiply with lower-triangular, row-normalised weights
W = torch.tril(torch.ones(T, T))                   # (T, T) = (6, 6)  lower-triangular ones
W = W / W.sum(1, keepdim=True)                      # (T, T) / (T, 1) = (T, T)  rows now sum to 1
mat = W @ X                                         # (T, T) @ (T, d) = (T, d)

check("loop average == matrix average", torch.allclose(loop, mat, atol=1e-6),
      f"max diff = {(loop-mat).abs().max():.1e}")
check("row 0 of W = [1,0,...] (token 0 sees only itself)",
      torch.allclose(W[0], torch.tensor([1.,0,0,0,0,0])), W[0].tolist())
check("row 4 of W = five 0.2's then 0", torch.allclose(W[4], torch.tensor([.2,.2,.2,.2,.2,0.])),
      [round(x,2) for x in W[4].tolist()])

fig, ax = plt.subplots(figsize=(6,5))
Chapter9.plot_attention(W.numpy(), title="Uniform causal averaging weights  W", ax=ax); plt.show()

---
## Part 3 · Making the weights smart — query, key, value

Uniform averaging weights *"the"* as heavily as *"river"*. We want each token to **decide** how much to pull from each other token, from content. Every token emits three vectors:

- **query** — "here's what I'm looking for" (*I'm* bank*, I want to know which kind*)
- **key** — "here's what I offer" (*I'm* river*, I offer geography*)
- **value** — "here's what I hand over if you attend to me"

The weight token *i* places on token *j* is **query·key** (a dot product): high match → high weight. Then, exactly as in Part 2, we take the weighted average — but of the **values**. `Q`, `K`, `V` come from learnable projections `W_q, W_k, W_v` (the linear layers of Chapter 4) — **the only things attention learns.**

### The formula

All of Part 3 collapses into one line — *the* equation of the transformer:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

Read it right-to-left in plain words: score every query against every key ($QK^{\top}$); divide by $\sqrt{d_k}$ (Part 4) to keep the numbers tame; `softmax` each row into weights that sum to 1; then use those weights to average the values $V$.

### What goes in, what comes out

Attention is a function from **a sequence of vectors to a sequence of vectors** — *same length in, same length out.* Put in $T$ tokens, get back $T$ tokens; each output token is a context-aware blend of the input values.

| Tensor | What it is | Shape |
|---|---|---|
| $X$ | the input — $T$ token vectors | $(T,\ d_{\text{model}})$ |
| $Q = X W_q$ | queries — *"what I'm looking for"* | $(T,\ d_k)$ |
| $K = X W_k$ | keys — *"what I offer"* | $(T,\ d_k)$ |
| $V = X W_v$ | values — *"what I hand over"* | $(T,\ d_v)$ |
| $QK^{\top}/\sqrt{d_k}$ | scores, every token vs every token | $(T,\ T)$ |
| $\text{softmax}(\cdot)$ | attention **weights**, each row sums to 1 | $(T,\ T)$ |
| **output** | weighted average of the values | $(T,\ d_v)$ |

Two shapes to burn in: the score/weight matrix is always $(T, T)$ — one number for every *(query, key)* pair — and the output is $(T, d_v)$ — **the same $T$ tokens you fed in, re-expressed.** (In self-attention $d_k = d_v = d_{\text{head}}$; a later projection maps $d_v$ back to $d_{\text{model}}$ so blocks can stack. Real models also carry a batch dimension $B$ and several heads at once — that's Part 6.)

> **A note on $d_k$ vs $d_v$.** These are separate knobs. $Q$ and $K$ *must* share a dimension — you take their dot product $QK^{\top}$ — so $d_q = d_k$ is a hard requirement. But $V$ only appears in $\text{weights} \times V$, so $d_v$ is free and the output simply inherits it. We set $d_k = d_v = d_{\text{head}}$ here by convention (as does the original paper), which keeps multi-head bookkeeping clean; some modern models (e.g. DeepSeek's Multi-head Latent Attention) deliberately make them differ.

**🔮 Predict first.** With 6 tokens, how big is the score matrix `Q @ Kᵀ`? After softmax, which axis must sum to 1 — rows or columns — and why?

In [ ]:
def self_attention(X, Wq, Wk, Wv, mask=None):
    # X (T, d_model);  Wq/Wk/Wv (d_model, d_head)
    Q = X @ Wq                                       # (T,d_model) @ (d_model,d_head) = (T, d_head)
    K = X @ Wk                                       # (T,d_model) @ (d_model,d_head) = (T, d_head)
    V = X @ Wv                                       # (T,d_model) @ (d_model,d_head) = (T, d_head)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])   # (T,d_head) @ (d_head,T) = (T,T), scaled (Part 4)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))   # (T,T), unchanged (Part 5)
    weights = torch.softmax(scores, dim=-1)          # (T,T)  each ROW sums to 1
    return weights @ V, weights                      # (T,T) @ (T,d_head) = (T,d_head);  weights (T,T)

d_model = dh = 8
Wq = torch.randn(d_model, dh) * 0.5                  # (d_model, d_head) = (8, 8)
Wk = torch.randn(d_model, dh) * 0.5                  # (d_model, d_head) = (8, 8)
Wv = torch.randn(d_model, dh) * 0.5                  # (d_model, d_head) = (8, 8)
Xp = torch.randn(T, d_model)                         # (T, d_model) = (6, 8)
out, w = self_attention(Xp, Wq, Wk, Wv)              # out (T, d_head) = (6, 8);  w (T, T) = (6, 6)

check("score/weight matrix is T×T", tuple(w.shape) == (T, T), tuple(w.shape))
check("output is (T, head_dim)", tuple(out.shape) == (T, dh), tuple(out.shape))
check("each query row sums to 1", torch.allclose(w.sum(-1), torch.ones(T), atol=1e-6),
      f"row sums ≈ {w.sum(-1).mean():.3f}")

fig, ax = plt.subplots(figsize=(6,5))
Chapter9.plot_attention(w.numpy(), title="Data-dependent attention weights (random init)", ax=ax)
plt.show()

---
## Part 4 · Softmax and √d scaling

The raw scores are just numbers. **Softmax** turns each row into a probability distribution over "which tokens to look at" — the same softmax that hid inside Chapter 2's cross-entropy. One wrinkle: we divide scores by **√(head dim)** *before* softmax. Dot products of `d`-dim vectors have variance ∝ `d`, so at large `d` the scores are huge, softmax **saturates** into a near one-hot spike, and attention collapses onto a single token.

**🔮 Predict first.** With no scaling and `d = 128`, will the softmax look more like uniform `[.25,.25,.25,.25]` or a spike `[.98,.01,...]`? Which is a worse starting point for learning (recall Chapter 4's flat, dead-gradient regions)?

In [ ]:
def peakiness(d_test, scale, n=2000):
    q = torch.randn(n, d_test)                       # (n, d)     one query per sample
    k = torch.randn(n, 5, d_test)                    # (n, 5, d)  five keys per sample
    s = (q.unsqueeze(1) @ k.transpose(-2, -1)).squeeze(1)  # (n,1,d) @ (n,d,5) = (n,1,5) -> (n, 5)
    if scale: s = s / math.sqrt(d_test)              # (n, 5)
    p = torch.softmax(s, -1)                          # (n, 5)  rows sum to 1
    return p.max(-1).values.mean().item(), s.std().item()

for dt in (2, 128):
    mx_ns, sd_ns = peakiness(dt, False); mx_s, sd_s = peakiness(dt, True)
    print(f"d={dt:>3}:  UNSCALED score-std={sd_ns:5.2f} max-p={mx_ns:.2f}   "
          f"|  SCALED score-std={sd_s:4.2f} max-p={mx_s:.2f}")

mx_unscaled, _ = peakiness(128, False); mx_scaled, _ = peakiness(128, True)
check("unscaled softmax saturates at high d", mx_unscaled > 0.8, f"max-p = {mx_unscaled:.2f}")
check("√d scaling keeps it soft", mx_scaled < mx_unscaled, f"{mx_scaled:.2f} < {mx_unscaled:.2f}")

torch.manual_seed(0)
q1 = torch.randn(128)                                # (128,) one query
ks = torch.randn(5, 128)                             # (5, 128) five keys
raw = ks @ q1                                        # (5, 128) @ (128,) = (5,)  raw scores
Chapter9.plot_softmax_comparison(
    {"unscaled (d=128)": torch.softmax(raw, 0).numpy(),
     "scaled by √d":     torch.softmax(raw/math.sqrt(128), 0).numpy()},
    title="Scaling stops softmax from collapsing onto one token"); plt.show()

---
## Part 5 · One line — masking, and the BERT/GPT fork

Now the fork in the road, and it's tiny. A **causal mask** sets the scores for *future* tokens to −∞ before softmax (−∞ → 0 weight). One line.

- **Keep the mask** → every token sees only the past → can predict the next token without cheating → **GPT** (generation).
- **Remove the mask** → every token sees the whole sentence, both directions → **BERT** (understanding). Useless for left-to-right generation — it's allowed to peek at the answer.

**BERT and GPT are the same machine.** The difference between "understands text" and "writes text" is whether you zero out the upper triangle of one matrix.

**🔮 Predict first.** In the masked heatmap, which entries are exactly 0? What shape does the colored region make?

In [ ]:
causal = torch.tril(torch.ones(T, T))            # (T, T) = (6, 6)  lower-triangular mask
_, w_masked   = self_attention(Xp, Wq, Wk, Wv, mask=causal)  # w_masked   (T, T)
_, w_unmasked = self_attention(Xp, Wq, Wk, Wv, mask=None)    # w_unmasked (T, T)

upper = torch.triu(torch.ones(T, T), 1).bool()   # (T, T) bool — the "future" entries
check("masked attention gives ZERO weight to the future",
      torch.allclose(w_masked[upper], torch.zeros(int(upper.sum())), atol=1e-7),
      f"max future weight = {w_masked[upper].max():.1e}")
check("unmasked attention gives NONZERO weight to the future",
      (w_unmasked[upper] > 0).all().item(), "all future weights > 0")

Chapter9.plot_attention_pair(w_masked.numpy(), w_unmasked.numpy()); plt.show()

---
## Part 6 · Multi-head attention

One head learns one notion of relevance. But a token may need several at once (what noun this adjective modifies **and** what verb governs this subject). **Multi-head attention** runs several small attention heads in parallel — each with its own `W_q,W_k,W_v` — then concatenates them. Same idea as Chapter 4's different hidden neurons learning different features.

**🔮 Predict first.** With `d_model = 64` and `8` heads, what is each head's dimension? Why shrink each head instead of running 8 full-width ones?

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, causal=False):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.dh, self.causal = n_head, d_model // n_head, causal
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)   # (d_model -> 3*d_model), all heads at once
        self.proj = nn.Linear(d_model, d_model, bias=False)       # (d_model -> d_model)

    def forward(self, x, return_weights=False):
        B, T, C = x.shape                            # x (B, T, C)  with C = d_model
        q, k, v = self.qkv(x).split(C, dim=-1)       # qkv(x) (B,T,3C) -> three (B, T, C)
        split = lambda t: t.view(B, T, self.n_head, self.dh).transpose(1, 2)  # (B,T,C)->(B,T,nh,dh)->(B,nh,T,dh)
        q = split(q)                                 # (B,T,C) -> (B, nh, T, dh)
        k = split(k)                                 # (B,T,C) -> (B, nh, T, dh)
        v = split(v)                                 # (B,T,C) -> (B, nh, T, dh)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)  # (B,nh,T,dh) @ (B,nh,dh,T) = (B, nh, T, T)
        if self.causal:
            att = att.masked_fill(torch.tril(torch.ones(T, T, device=x.device)) == 0, float("-inf"))  # (B,nh,T,T)
        att = torch.softmax(att, dim=-1)             # (B, nh, T, T)  rows sum to 1
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)  # (B,nh,T,T)@(B,nh,T,dh)=(B,nh,T,dh) -> concat (B,T,C)
        out = self.proj(out)                          # (B,T,C) @ (C,C) = (B, T, C)
        return (out, att) if return_weights else out

mha = MultiHeadAttention(64, 8)
out6, att6 = mha(torch.randn(1, T, 64), return_weights=True)  # in (1,T,64) -> out6 (1,T,64);  att6 (1, 8, T, T)
check("output keeps d_model = 64", out6.shape[-1] == 64, out6.shape[-1])
check("each of 8 heads has dim 64/8 = 8", mha.dh == 8, mha.dh)
check("different heads → different attention patterns",
      (att6[0,0]-att6[0,1]).abs().max() > 0.01, f"max diff = {(att6[0,0]-att6[0,1]).abs().max():.3f}")

Chapter9.plot_heads([att6[0,0].detach().numpy(), att6[0,3].detach().numpy()],
                    tokens=[f"t{i}" for i in range(T)]); plt.show()

---
## Part 7 · The transformer block — attention meets Chapter 4

Attention lets tokens **share** information; it doesn't **think**. After gathering context, each token is processed by a **feed-forward network** — and this is *literally the 2-layer network from Chapter 4* (Linear → nonlinearity → Linear), applied per token. Attention gathers; the MLP thinks.

Two pieces of plumbing make deep stacks trainable, both earned earlier:
- **Residual connections** `x = x + block(x)` — the block only learns a *correction*, keeping gradients alive (Chapter 4's gradient-flow concern).
- **LayerNorm** — keeps each token's numbers well-conditioned across many layers.

**🔮 Predict first.** If a block's input is `(tokens=6, dim=64)`, what's its output shape — and why *must* it match if we want to stack blocks?

In [ ]:
class FeedForward(nn.Module):
    "The Chapter-4 two-layer net, applied to each token independently."
    def __init__(self, d_model, mult=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, mult*d_model), nn.GELU(),  # (d -> 4d)
                                 nn.Linear(mult*d_model, d_model))             # (4d -> d)
    def forward(self, x): return self.net(x)         # (B,T,d) -> (B,T,4d) -> (B, T, d)

class Block(nn.Module):
    def __init__(self, d_model, n_head, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = MultiHeadAttention(d_model, n_head, causal)
        self.ln2 = nn.LayerNorm(d_model); self.ff = FeedForward(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # (B,T,C) + (B,T,C) = (B,T,C)  residual around attention
        x = x + self.ff(self.ln2(x))     # (B,T,C) + (B,T,C) = (B,T,C)  residual around the MLP
        return x

blk = Block(64, 8)
xb = torch.randn(2, T, 64)               # (B, T, C) = (2, 6, 64)
yb = blk(xb)                             # (B, T, C) = (2, 6, 64)  same shape in and out
check("block output shape == input shape", yb.shape == xb.shape, tuple(yb.shape))

# residual proof: zero the sub-layer outputs and the block becomes the identity
with torch.no_grad():
    blk.attn.proj.weight.zero_(); blk.ff.net[-1].weight.zero_(); blk.ff.net[-1].bias.zero_()
check("with zeroed sub-layers, residual makes the block ≈ identity",
      torch.allclose(blk(xb), xb, atol=1e-5), f"max diff = {(blk(xb)-xb).abs().max():.1e}")

---
## Part 8 · Stack it → the Chapter 8 contextual embeddings

Stack a few **unmasked** blocks and you have a miniature BERT: token embedding + positional embedding + blocks. Read out the final-layer vectors — those are **contextual embeddings**, computed by the mechanism you just built.

**One piece we owe you: position.** Attention is a weighted average, so it's **order-blind** ("river bank" and "bank river" would give the same weights). Real transformers add a **positional embedding** to each token first, so the model knows *where* each token sits. We include it here.

**🔮 Predict first.** The *static* embedding of *"bank"* is one fixed vector. After passing "the river bank" and "the money bank" through the blocks, will the two *"bank"* vectors be identical or move apart? (This is the Chapter 8 result — now you know why.)

In [ ]:
vocab = ["the", "river", "bank", "money", "gave", "loan", "flowed"]
w2i = {w: i for i, w in enumerate(vocab)}

class Encoder(nn.Module):
    def __init__(self, V, d_model=32, n_head=4, n_layer=3, max_T=8):
        super().__init__()
        self.tok = nn.Embedding(V, d_model)          # (V -> d)     token lookup table
        self.pos = nn.Embedding(max_T, d_model)      # (max_T -> d) position lookup table
        self.blocks = nn.ModuleList([Block(d_model, n_head, causal=False) for _ in range(n_layer)])
        self.ln = nn.LayerNorm(d_model)
    def forward(self, idx):
        T = idx.shape[1]                             # idx (B, T) of token ids
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))  # (B,T,d) + (T,d) = (B, T, d)
        for b in self.blocks: x = b(x)               # (B,T,d) -> (B, T, d)
        return self.ln(x)                             # (B, T, d)

torch.manual_seed(1)
enc = Encoder(len(vocab))
sent = lambda ws: torch.tensor([[w2i[w] for w in ws]])  # list[str] -> (1, len) ids
h_river = enc(sent(["the","river","bank"]))[0, 2]   # (1,3,d) -> [0,2] = (d,)  'bank' with river
h_money = enc(sent(["the","money","bank"]))[0, 2]   # (d,)  'bank' with money

check("static 'bank' is a single context-free vector",
      enc.tok.weight[w2i['bank']].shape == h_river.shape, tuple(h_river.shape))
check("contextual 'bank' DIFFERS across the two sentences",
      (h_river - h_money).norm() > 1e-3, f"dist = {(h_river-h_money).norm():.3f}")

# Direction check with an interpretable averaging attention (value = identity):
emb = enc.tok.weight.detach()                        # (V, d) static embedding table
avg = lambda ws: torch.stack([emb[w2i[w]] for w in ws]).mean(0)  # stack (len,d) -> mean(0) = (d,)
cos = lambda a, b: F.cosine_similarity(a, b, 0).item()
c_r = cos(avg(["the","river","bank"]), emb[w2i["river"]])
c_m = cos(avg(["the","money","bank"]), emb[w2i["river"]])
check("attention pulls 'bank' toward its context (higher cos-sim to 'river' when 'river' is present)",
      c_r > c_m, f"cos(bank|river, river)={c_r:.2f} > cos(bank|money, river)={c_m:.2f}")

fig, ax = plt.subplots(figsize=(7,4))
labels = ["cos( bank | river , river )", "cos( bank | money , river )"]
ax.bar(labels, [c_r, c_m], color=["#2E86AB", "#C1121F"], edgecolor="black")
ax.axhline(0, color="gray", lw=0.8); ax.set_ylabel("cosine similarity", weight="bold")
ax.set_title("'bank' leans toward whichever context word is present", weight="bold")
plt.tight_layout(); plt.show()

---
## Part 9 · The payoff — a tiny GPT that beats the bigram

Put the **causal mask** back in, add a final linear layer to a distribution over the next character, and train on the names with **Chapter 2's cross-entropy** and **Chapter 4's gradient descent**. That's a GPT.

First, the baseline: the **bigram loss** is the Chapter-3 floor — the best you can do with one character of memory.

**🔮 Predict first.** The bigram produced runaway garbage like *"kanocikhilinileea"* because it only remembers one character. Your GPT sees the whole prefix. Which kind of failure should nearly vanish? Will the loss drop below the bigram floor?

In [ ]:
names = Chapter9.load_names()
chars = ["."] + sorted(set("".join(names)))          # '.' = start/end token (index 0)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
V = len(chars)
print(f"{len(names):,} names, vocab = {V}")

# --- bigram baseline loss (the Chapter-3 floor), from smoothed counts ---
Nmat = torch.ones(V, V)                              # (V, V) bigram counts (+1 smoothing)
for wd in names:
    ids = [0] + [stoi[c] for c in wd] + [0]
    for a, b in zip(ids, ids[1:]): Nmat[a, b] += 1
Pbig = Nmat / Nmat.sum(1, keepdim=True)              # (V,V) / (V,1) = (V, V)  rows are P(next | prev)
nll = cnt = 0.0
for wd in names:
    ids = [0] + [stoi[c] for c in wd] + [0]
    for a, b in zip(ids, ids[1:]): nll += -torch.log(Pbig[a, b]); cnt += 1
bigram_loss = (nll / cnt).item()
print(f"bigram baseline loss = {bigram_loss:.4f} nats")

In [ ]:
# --- data as one long stream of ids; batches of (context -> next char) ---
block_size = 16
stream = []
for wd in names: stream += [0] + [stoi[c] for c in wd]
stream.append(0)
stream = torch.tensor(stream, dtype=torch.long)      # (N,)  one long stream of ids
cut = int(0.9 * len(stream))
train_data = stream[:cut]                            # (~0.9N,)
val_data   = stream[cut:]                            # (~0.1N,)

def get_batch(split, bs=64):
    dd = train_data if split == "train" else val_data
    ix = torch.randint(len(dd) - block_size - 1, (bs,))          # (bs,) random start offsets
    xb = torch.stack([dd[i:i+block_size] for i in ix])           # (bs, block_size)  contexts
    yb = torch.stack([dd[i+1:i+block_size+1] for i in ix])       # (bs, block_size)  next-char targets
    return xb.to(DEVICE), yb.to(DEVICE)

class GPT(nn.Module):
    def __init__(self, V, d_model=64, n_head=4, n_layer=3, block_size=16):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(V, d_model)          # (V -> d)          token table
        self.pos = nn.Embedding(block_size, d_model) # (block_size -> d) position table
        self.blocks = nn.ModuleList([Block(d_model, n_head, causal=True) for _ in range(n_layer)])
        self.ln = nn.LayerNorm(d_model); self.head = nn.Linear(d_model, V)   # head: (d -> V)
    def forward(self, idx, targets=None):
        T = idx.shape[1]                             # idx (B, T) of token ids
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))  # (B,T,d) + (T,d) = (B, T, d)
        for b in self.blocks: x = b(x)               # (B,T,d) -> (B, T, d)
        logits = self.head(self.ln(x))               # (B,T,d) @ (d,V) = (B, T, V)
        loss = None if targets is None else F.cross_entropy(
            logits.view(-1, V), targets.view(-1))    # (B*T, V) vs (B*T,) -> scalar
        return logits, loss
    @torch.no_grad()
    def generate(self):
        idx = torch.zeros(1, 1, dtype=torch.long, device=next(self.parameters()).device)  # (1, 1) running context
        out = []
        for _ in range(120):
            logits, _ = self(idx[:, -self.block_size:])  # last ≤block_size ids -> logits (1, T, V)
            nxt = torch.multinomial(torch.softmax(logits[:, -1, :], -1), 1)  # logits[:,-1,:] (1,V) -> nxt (1, 1)
            if nxt.item() == 0: break
            out.append(itos[nxt.item()]); idx = torch.cat([idx, nxt], 1)  # (1,T) ++ (1,1) = (1, T+1)
        return "".join(out)

model = GPT(V, block_size=block_size).to(DEVICE)
print(f"GPT parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# --- train (gradient descent from Chapter 4) ---
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
losses, t0 = [], time.time()
for step in range(3000):
    xb, yb = get_batch("train")                      # xb (bs, block_size);  yb (bs, block_size)
    _, loss = model(xb, yb)                           # loss: scalar
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % 50 == 0: losses.append(loss.item())
    if step % 500 == 0: print(f"step {step:4d}   loss {loss.item():.4f}")
print(f"trained 3000 steps in {time.time()-t0:.1f}s on {DEVICE}")

In [ ]:
# --- evaluate and compare to the bigram floor ---
model.eval()
with torch.no_grad():
    val_loss = torch.stack([model(*get_batch("val"))[1] for _ in range(50)]).mean().item()
check("the transformer beats the bigram floor",
      val_loss < bigram_loss, f"val {val_loss:.3f} < bigram {bigram_loss:.3f} nats")

Chapter9.plot_loss(losses, baseline=bigram_loss, steps_per_point=50,
                   title="Char-level GPT training loss vs the Chapter-3 bigram floor"); plt.show()

In [ ]:
# --- generate names, side by side with a bigram sampler (same generation loop as Ch 3) ---
def bigram_sample():
    ix, out = 0, []
    for _ in range(30):
        ix = torch.multinomial(Pbig[ix], 1).item()   # Pbig[ix] (V,) -> sample -> int
        if ix == 0: break
        out.append(itos[ix])
    return "".join(out)

torch.manual_seed(0); gpt_names = [model.generate() for _ in range(15)]
torch.manual_seed(0); big_names = [bigram_sample() for _ in range(15)]
print(Chapter9.two_column_names(big_names, gpt_names))

---
## Summary

**The one idea.** A contextual embedding is a **weighted average** of other tokens' vectors; self-attention **computes the weights** from the tokens' own content, fresh for every sequence.

**How.** Each token emits a **query**, **key**, **value** (the only learned parameters). score = query·key, **scaled by √d**, **softmax**'d into a distribution, then applied to the **values**. A weighted average over a sequence is a **matrix multiply**.

**The BERT/GPT fork — one line.** Mask the upper triangle → every token sees only the past → **GPT** (generation). Leave it → both directions → **BERT** (understanding). Same machine.

**The plumbing (all earned earlier).** *Multi-head* = several specialists in parallel (Ch 4 neurons). *Feed-forward* = the Ch 4 two-layer net, per token. *Residuals* keep gradients alive; *LayerNorm* keeps numbers sane; *positional embeddings* because attention is order-blind. A **block** maps a sequence of vectors to a same-shaped one, so you can **stack** it.

**Where it landed.** Unmasked stack → the **Chapter 8** contextual embeddings, mechanism visible. Masked stack + next-token head → a **GPT** that crushed the **Chapter 3** bigram floor (val ≈ 1.87 vs 2.45 nats) and generates far more name-like names — same dataset, same generation loop, wall gone.

**What's next:** scale (more blocks, data, compute), real subword tokenization (Ch 3's characters → tokens), and the training regimes — pretraining, fine-tuning, alignment — that turn a next-token predictor into an assistant.

---
## Discussion Questions

1. **The whole trick in one sentence.** In what sense is a static word2vec embedding (Ch 6) the *degenerate* case of a contextual one — an average with what weights?
2. **Why data-dependent weights?** Uniform averaging (Part 2) needs zero learned parameters. Give a two-sentence example where it produces a clearly worse *"bank"* vector than query/key attention, and say which words should have gotten more or less weight.
3. **The mask as a limit.** Removing the mask lets BERT see the future and understand better — yet we can't use it to generate left-to-right. Precisely what goes wrong if you try?
4. **Scaling & saturation.** When softmax saturates to near one-hot (Part 4), what happens to the gradients into `W_q`, `W_k`, and why does that stall learning? Connect to Chapter 4's flat loss regions.
5. **Heads vs. depth.** More heads per block, or more blocks? What does each buy you?
6. **Position matters.** Construct two sentences that are identical as a *set* of words but mean different things, such that a position-blind model is forced to represent them identically.
7. **Closing the Chapter 3 loop.** The bigram had a hard loss floor; the transformer blew past it. Is there any prediction the bigram might make *better*, at least early in training? What did the bigram get "for free" that the transformer has to learn?